Хочу получить датасет из названий фильмов, их описаний и списков их жанров

In [1]:
import kagglehub
import pandas as pd
import os
import json

from sklearn.preprocessing import MultiLabelBinarizer


In [2]:
path = kagglehub.dataset_download("tmdb/tmdb-movie-metadata")

print("Path to dataset files:", path)

Path to dataset files: /home/slonik11111111/.cache/kagglehub/datasets/tmdb/tmdb-movie-metadata/versions/2


In [3]:
movies_dataset_path = os.path.join(path, "tmdb_5000_movies.csv")
movies_dataset = pd.read_csv(movies_dataset_path)
movies_dataset = movies_dataset[["title", "overview", "genres"]].dropna()
movies_dataset.head(5)

,title,overview,genres
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam..."
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam..."


Нужна функция распарсить json в нормальный вид (в список из жанров)

In [4]:
def parse_json_with_genres(json_string):
    try :
        genres_list = json.loads(json_string)
        return [genre['name'] for genre in genres_list]
    except:
        return []


In [5]:
movies_dataset["clean_genres"] = movies_dataset["genres"].apply(parse_json_with_genres)
movies_dataset = movies_dataset[movies_dataset['clean_genres'].map(len) > 0]
movies_dataset.head(5)

,title,overview,genres,clean_genres
0,Avatar,"In the 22nd century, a paraplegic Marine is di...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[Action, Adventure, Fantasy, Science Fiction]"
1,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...","[Adventure, Fantasy, Action]"
2,Spectre,A cryptic message from Bond’s past sends him o...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[Action, Adventure, Crime]"
3,The Dark Knight Rises,Following the death of District Attorney Harve...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...","[Action, Crime, Drama, Thriller]"
4,John Carter,"John Carter is a war-weary, former military ca...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[Action, Adventure, Science Fiction]"


Теперь надо превратить список жанров в колонки с 0 и 1 и сохранить датасет

In [6]:
mlb = MultiLabelBinarizer()
genres_encoded = mlb.fit_transform(movies_dataset["clean_genres"])

genres_df = pd.DataFrame(genres_encoded, columns=mlb.classes_, index=movies_dataset.index)


model_data = pd.concat([movies_dataset[["title","overview"]], genres_df], axis=1)

model_data.to_csv("data.csv", index=False)
